# Topic Analysis

In case of wanting to test the code yourself, run this notebook in Google Collab with the GPU units on. This code is designed to run in computers that support CUDA, therefore it must be run in Google Collab if you are utilizing a Macbook.

The other notebook named topic_google_collab.ipynb contains the cell outputs from the execution.

In [ ]:
!pip install transformers==4.57.6
!pip install simpletransformers==0.70.5

In [23]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import classification_report
from simpletransformers.classification import ClassificationModel, ClassificationArgs
import matplotlib.pyplot as plt
import seaborn as sn

/opt/anaconda3/envs/lab6-text-mining/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Custom Dataset Generation

In [30]:
imdb_path = 'topic-datasets/IMDB Dataset.csv'
kindle_path = 'topic-datasets/preprocessed_kindle_review .csv'

# Load datasets
imdb_full = pd.read_csv(imdb_path)
kindle_full = pd.read_csv(kindle_path)
restaurant_full = pd.read_csv('topic-datasets/Restaurant_Reviews.tsv', sep='\t')
test_raw = pd.read_csv('topic-datasets/Sentiment-topic-test.tsv', sep='\t')
topic_mapping = { 'movie': 0, 'book': 1, 'restaurant': 2}

# Sample exactly 1000 random instances reproducibly for each class
imdb_sample = imdb_full.sample(n=1000, random_state=42)
kindle_sample = kindle_full.sample(n=1000, random_state=42)
restaurant_sample = restaurant_full.sample(n=1000, random_state=42)

#Keep relevant columns for topic classification task
imdb_clean = pd.DataFrame({'text': imdb_sample['review'],  'labels': 0 })
kindle_clean = pd.DataFrame({ 'text': kindle_sample['reviewText'], 'labels': 1 })
restaurant_clean = pd.DataFrame({ 'text': restaurant_sample['Review'], 'labels': 2 })

all_training_data = pd.concat([imdb_clean, kindle_clean, restaurant_clean], ignore_index=True)
train = all_training_data.sample(frac=1, random_state=42).reset_index(drop=True)
train = train.dropna().reset_index(drop=True)

print(train.head())
print("\nClass distribution:")
print(train['labels'].value_counts())


                                                text  labels
0  This is a pretty good story,has a little of ev...       1
1  This is not a book to be set aside lightly. Th...       1
2  This is one of the most dissapointing purchase...       1
3  Alexander Nevsky (1938) is a brilliant piece o...       0
4  Insults, profound deuchebaggery, and had to go...       2

Class distribution:
labels
1    1000
0    1000
2    1000
Name: count, dtype: int64


# Converting Dataset to Compatible (Simpletransformers) Format

In [22]:
from sklearn.model_selection import train_test_split

test = pd.DataFrame({ 'text': test_raw['text'], 'labels': test_raw['topic'].map(topic_mapping)})
train, dev = train_test_split(train, test_size=0.1, random_state=42, stratify=train['labels'])

print(test.head())


                                                text  labels
0  It took eight years for Warner Brothers to rec...       0
1  All the New York University students love this...       2
2  This Italian place is really trendy but they h...       2
3  In conclusion, my review of this book would be...       1
4  The story of this movie is focused on Carl Bra...       0


# Model Parameters and Training

In [ ]:
model_args = ClassificationArgs()
model_args.overwrite_output_dir=True
model_args.evaluate_during_training=True
model_args.num_train_epochs=10
model_args.train_batch_size=32
model_args.learning_rate=4e-6
model_args.max_seq_length=256
model_args.use_early_stopping=True
model_args.early_stopping_delta=0.01
model_args.early_stopping_metric='eval_loss'
model_args.early_stopping_metric_minimize=True
model_args.early_stopping_patience=2
model_args.evaluate_during_training_steps=32

steps_per_epoch = int(np.ceil(len(train) / float(model_args.train_batch_size)))
print('Each epoch will have {:,} steps.'.format(steps_per_epoch))

In [ ]:
model = ClassificationModel('bert', 'bert-base-cased', num_labels=3, args=model_args, use_cuda=True)

In [ ]:
print(str(model.args).replace(',', '\n'))

# Model Fine-Tuning and Evaluation

In [ ]:
_, history = model.train_model(train, eval_df=dev)

In [ ]:
result, model_outputs, wrong_predictions = model.eval_model(dev)
result

In [ ]:
predicted, probabilities = model.predict(test.text.to_list())
test['predicted'] = predicted
pd.set_option('display.max_rows', None)
print(test)

In [ ]:
print(classification_report(test['labels'], test['predicted']))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         2
           2       1.00      1.00      1.00         3

    accuracy                           1.00        10
    macro avg       1.00      1.00      1.00        10
    weighted avg       1.00      1.00      1.00        10

# Testing Tricky Sentences

In [ ]:
difficult_sentences = [
    # Has "menu" (restaurant) but is actually about a movie
    "The director left the script on the menu at the diner.",
    # Uses food adjectives to describe a book
    "The plot of this novel was spicy, but the ending was severely undercooked.",
    # Has "reading" (book) but is about a restaurant
    "I was reading the chalkboard outside to see what the chef's specials were." ]

difficult_predictions, _ = model.predict(difficult_sentences)

# Map back to words
inverse_mapping = {0: 'movie', 1: 'book', 2: 'restaurant'}
for sentence, pred in zip(difficult_sentences, difficult_predictions):
    print(f"Prediction: {inverse_mapping[pred]:<10} | Sentence: {sentence}")